# P8 — Interpretabilidad: Grad-CAM y mapa espacial

In [4]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## Grad-CAM y localización sobre imágenes procesadas

P8 utiliza los CSV de parches producidos por P6 y P7. Para cada imagen recupera la ROI con mayor probabilidad, calcula Grad-CAM sobre ese parche y exporta la visualización y un CSV trazable.


In [ ]:
import torch, pandas as pd, numpy as np
from PIL import Image, ImageOps, ImageDraw
from src.models import build_resnet50
from src.explainability import gradcam_patch
from src.inference import probability_map
import matplotlib.pyplot as plt

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=build_resnet50(cfg['model']['head_hidden'],cfg['model']['dropout'],pretrained=False).to(device)
model.load_state_dict(torch.load(ROOT/'models/resnet50_finetuned.pt',map_location=device)); model.eval()
GRAD=ROOT/'results/gradcam'; GRAD.mkdir(parents=True,exist_ok=True)
HEAT=ROOT/'results/heatmaps'; HEAT.mkdir(parents=True,exist_ok=True)
print('Modelo cargado:',device)


In [ ]:
records=[]

def process_from_patch_csv(image_path, patch_csv, group, image_prob=None):
    patch_df=pd.read_csv(patch_csv)
    if patch_df.empty: return None
    best=patch_df.loc[patch_df['prob_anomaly'].idxmax()]
    out=GRAD/f'{group}__{Path(image_path).stem}_gradcam.png'
    hout=HEAT/f'{group}__{Path(image_path).stem}_probability_map.png'
    if out.exists() and hout.exists():
        return {"group":group,"file":Path(image_path).name,"image_probability":image_prob,"max_patch_prob":float(best.prob_anomaly),
                "gradcam_roi_prob":float(best.prob_anomaly),"x1":int(best.x1),"y1":int(best.y1),"x2":int(best.x2),"y2":int(best.y2),
                "gradcam_file":str(out.relative_to(ROOT)),"heatmap_file":str(hout.relative_to(ROOT))}
    full=ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    box=(int(best.x1),int(best.y1),int(best.x2),int(best.y2))
    patch=full.crop(box)
    ps=int(cfg['field_inference']['patch_size'])
    if patch.size!=(ps,ps):
        canvas=Image.new('RGB',(ps,ps)); canvas.paste(patch,(0,0)); patch=canvas
    cam,overlay,roi_prob=gradcam_patch(model,patch,device,input_size=cfg['model']['input_size'],alpha=cfg['field_inference']['heatmap_alpha'])
    preview=full.copy(); ImageDraw.Draw(preview).rectangle(box,outline='red',width=max(2,full.width//500))

    # 4 paneles: imagen, ROI, CAM, overlay
    fig,axs=plt.subplots(1,4,figsize=(10,2.6))
    axs[0].imshow(preview); axs[0].set_title('Imagen + ROI',fontsize=8)
    axs[1].imshow(patch); axs[1].set_title(f'ROI | P={roi_prob:.3f}',fontsize=8)
    axs[2].imshow(cam,cmap='jet',vmin=0,vmax=1); axs[2].set_title('Grad-CAM',fontsize=8)
    axs[3].imshow(overlay); axs[3].set_title('Superposición',fontsize=8)
    for ax in axs: ax.axis('off')
    fig.suptitle(f'{group} | {Path(image_path).name}',fontsize=9); fig.tight_layout()
    fig.savefig(out,dpi=200,bbox_inches='tight'); plt.close(fig)

    # mapa espacial de probabilidad de patches
    hmap=probability_map(full.size,patch_df)
    fig,ax=plt.subplots(figsize=(5.5,4)); ax.imshow(full); ax.imshow(hmap,cmap='jet',alpha=cfg['field_inference']['heatmap_alpha'],vmin=0,vmax=1)
    ax.set_title(f'{Path(image_path).name} | mapa de probabilidad'); ax.axis('off'); fig.tight_layout()
    fig.savefig(hout,dpi=180,bbox_inches='tight'); plt.close(fig)

    return {"group":group,"file":Path(image_path).name,"image_probability":image_prob,"max_patch_prob":float(best.prob_anomaly),
            "gradcam_roi_prob":float(roi_prob),"x1":int(best.x1),"y1":int(best.y1),"x2":int(best.x2),"y2":int(best.y2),
            "gradcam_file":str(out.relative_to(ROOT)),"heatmap_file":str(hout.relative_to(ROOT))}

# Campo/prueba: CSV producidos por P6
field_summary_path=ROOT/'results/inference/field_qualitative_results.csv'
if field_summary_path.exists():
    fs=pd.read_csv(field_summary_path)
    for _,r in fs.iterrows():
        ip=ROOT/'pictures'/r['group']/r['file']
        pc=ROOT/'results/inference/field_patches'/f"{r['group']}__{Path(r['file']).stem}_patches.csv"
        if ip.exists() and pc.exists():
            rec=process_from_patch_csv(ip,pc,r['group'],float(r['prob_anomaly']))
            if rec: records.append(rec)


grad_summary=pd.DataFrame(records)
grad_summary.to_csv(GRAD/'gradcam_summary.csv',index=False)
print('Grad-CAM de campo generados:',len(grad_summary)); display(grad_summary)


In [ ]:
import shutil
out=GRAD/'field_gradcam_mosaic.png'
pdf_out=GRAD/'field_gradcam_mosaic.pdf'
if False:
    pass
else:
    preferred=['campo_004.jpg','campo_007.jpg','campo_012.jpg']
    sel=grad_summary[grad_summary['file'].isin(preferred)].copy()
    if len(sel)<3:
        sel=(grad_summary[grad_summary['group'].isin(['anomaly_field','test'])]
             .sort_values('max_patch_prob',ascending=False).head(3).copy())

    fig,axs=plt.subplots(len(sel),4,figsize=(9.2,2.25*len(sel)))
    axs=np.atleast_2d(axs)
    for row_idx,(_,r) in enumerate(sel.iterrows()):
        group=r['group']; image_path=ROOT/'pictures'/group/r['file']
        patch_csv=ROOT/'results/inference/field_patches'/f"{group}__{Path(r['file']).stem}_patches.csv"
        pdf=pd.read_csv(patch_csv); best=pdf.loc[pdf.prob_anomaly.idxmax()]
        full=ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
        box=(int(best.x1),int(best.y1),int(best.x2),int(best.y2)); patch=full.crop(box)
        ps=int(cfg['field_inference']['patch_size'])
        if patch.width < ps or patch.height < ps:
            canvas=Image.new('RGB',(ps,ps)); canvas.paste(patch,(0,0)); patch=canvas
        cam,overlay,roi_prob=gradcam_patch(model,patch,device,input_size=cfg['model']['input_size'],alpha=cfg['field_inference']['heatmap_alpha'])
        prev=full.copy(); ImageDraw.Draw(prev).rectangle(box,outline='red',width=max(2,full.width//500))
        panels=[prev,patch,cam,overlay]; titles=[r['file'],f"ROI P={roi_prob:.3f}",'Grad-CAM','Superposición']
        for j,(obj,title) in enumerate(zip(panels,titles)):
            axs[row_idx,j].imshow(obj,cmap='jet' if j==2 else None,vmin=0 if j==2 else None,vmax=1 if j==2 else None)
            axs[row_idx,j].set_title(title,fontsize=7,pad=4); axs[row_idx,j].axis('off')
    fig.tight_layout(h_pad=1.15,w_pad=.5)
    out=GRAD/'field_gradcam_mosaic.png'; fig.savefig(out,dpi=220,bbox_inches='tight')
    pdf_out=GRAD/'field_gradcam_mosaic.pdf'; fig.savefig(pdf_out,bbox_inches='tight'); plt.close(fig)
    sel.to_csv(GRAD/'field_gradcam_examples.csv',index=False)
    # La figura del paper se actualiza directamente desde P8
    shutil.copy2(out,ROOT/'results/paper/fig7_campo.png')
    shutil.copy2(pdf_out,ROOT/'results/paper/fig7_campo.pdf')
    print('P8 exportó Grad-CAM, heatmaps, CSV de trazabilidad y mosaico del artículo.')


In [ ]:
import shutil
legacy = ROOT / "paper/figures"
outdir = ROOT / "results/paper"; outdir.mkdir(parents=True, exist_ok=True)
for ext in ["png", "pdf", "svg"]:
    src = legacy / f"fig6_gradcam.{ext}"
    if src.exists(): shutil.copy2(src, outdir / src.name)
print("P8 publicó fig6_gradcam en results/paper.")
